# Gaussian Discriminant Analysis

> Based on *Lecture Notes* — Andrew Ng

GDA is a **generative learning algorithm** that models the class-conditional distributions $p(x \mid y)$ as multivariate Gaussians, then applies Bayes' rule to compute $p(y \mid x)$.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Distinguish** generative from discriminative models and explain the trade-offs
2. **Derive** the MLE parameter estimates for GDA from the joint log-likelihood
3. **Implement** GDA from scratch and visualize the linear decision boundary
4. **Explain** why GDA implies logistic regression (but not vice versa)
5. **Compare** LDA (shared $\Sigma$) with QDA (separate $\Sigma_k$) and identify when each is appropriate

> **Prerequisites**: Multivariate Gaussian, MLE, linear algebra (covariance matrices, matrix inverse).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression

np.random.seed(42)

## 1. Generative vs Discriminative Models

| | Discriminative | Generative |
|---|---|---|
| **What is modelled** | $p(y \mid x)$ directly | $p(x \mid y)$ and $p(y)$ separately |
| **Examples** | Logistic regression, SVM, neural nets | GDA, Naive Bayes |
| **At prediction** | Use $p(y \mid x)$ directly | Bayes' rule: $p(y\mid x) \propto p(x\mid y)\,p(y)$ |
| **Assumption strength** | Fewer | Stronger (distributional form) |

**Intuition** — classifying tumours as malignant/benign:
- *Discriminative*: learn a boundary in feature space that separates the two classes
- *Generative*: model what benign tumours look like and what malignant tumours look like separately; a new tumour is assigned to whichever class it more likely came from

**Key trade-off:** if the generative assumption (e.g. Gaussian) holds, GDA needs fewer examples to reach the same accuracy as logistic regression. If it doesn't hold, logistic regression is more robust.

## 2. The Multivariate Gaussian

$x \in \mathbb{R}^n$ follows $\mathcal{N}(\mu, \Sigma)$ if:

$$p(x;\,\mu,\Sigma) = \frac{1}{(2\pi)^{n/2}|\Sigma|^{1/2}} \exp\!\left(-\frac{1}{2}(x-\mu)^T \Sigma^{-1} (x-\mu)\right)$$

**Parameters:**

| Symbol | Shape | Meaning |
|---|---|---|
| $\mu$ | $\mathbb{R}^n$ | Mean — centre of the distribution |
| $\Sigma$ | $\mathbb{S}^n_{++}$ | Covariance — shape and orientation of the probability ellipsoid |

**Effect of $\Sigma$ on the shape:**

| $\Sigma$ | Shape of contours |
|---|---|
| $\sigma^2 I$ | Circles (isotropic) |
| Diagonal | Axis-aligned ellipses |
| Full (off-diagonal $\neq 0$) | Rotated ellipses |

**Key identity** — the exponent is a quadratic form called the **Mahalanobis distance**:

$$d_M(x, \mu)^2 = (x-\mu)^T \Sigma^{-1} (x-\mu)$$

Contours of constant probability are ellipsoids $\{x : d_M(x,\mu) = c\}$.

In [ ]:
# Effect of covariance structure on the Gaussian shape
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
mu = np.array([0., 0.])
grid_1d = np.linspace(-4, 4, 200)
xx, yy = np.meshgrid(grid_1d, grid_1d)
pos = np.stack([xx, yy], axis=-1)

configs = [
    (np.array([[1., 0.], [0., 1.]]),    "Isotropic\n$\\Sigma = I$"),
    (np.array([[2., 0.], [0., 0.5]]),   "Axis-aligned\n$\\Sigma = \\mathrm{diag}(2, 0.5)$"),
    (np.array([[1., 0.8], [0.8, 1.]]),  "Correlated\n$\\Sigma_{12} = 0.8$"),
]

for ax, (Sigma, title) in zip(axes, configs):
    rv = multivariate_normal(mu, Sigma)
    Z  = rv.pdf(pos)
    ax.contourf(xx, yy, Z, levels=12, cmap="Blues")
    ax.contour(xx, yy, Z, levels=6, colors="white", alpha=0.6, linewidths=0.8)
    ax.plot(0, 0, "r+", ms=12, mew=2)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
    ax.set_aspect("equal")

plt.suptitle("Multivariate Gaussian: effect of $\\Sigma$", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. The GDA Model

For binary classification ($y \in \{0, 1\}$), GDA assumes:

**Prior:**
$$y \sim \text{Bernoulli}(\phi) \quad \Rightarrow \quad p(y=1) = \phi,\quad p(y=0) = 1-\phi$$

**Class-conditional distributions (shared covariance):**
$$x \mid y=0 \sim \mathcal{N}(\mu_0, \Sigma)$$
$$x \mid y=1 \sim \mathcal{N}(\mu_1, \Sigma)$$

**Parameters to learn:** $\phi,\ \mu_0,\ \mu_1,\ \Sigma$

**Example** — two classes in $\mathbb{R}^2$:

| Parameter | Value | Meaning |
|---|---|---|
| $\phi = 0.4$ | 40 % of examples are class 1 | class-1 prior |
| $\mu_0 = (-1, -1)$ | centre of class 0 | class-0 mean |
| $\mu_1 = (2, 1)$ | centre of class 1 | class-1 mean |
| $\Sigma = \begin{bmatrix}1.5 & 0.5\\0.5 & 1\end{bmatrix}$ | shared shape | both classes have the same covariance |

**Key assumption:** sharing $\Sigma$ across both classes leads to a **linear** decision boundary (LDA). Allowing separate $\Sigma_0, \Sigma_1$ gives a **quadratic** boundary (QDA).

## 4. MLE Parameter Estimation

**Joint log-likelihood** over $m$ i.i.d. training examples $\{(x^{(i)}, y^{(i)})\}$:

$$\ell = \sum_{i=1}^{m} \log p(x^{(i)}, y^{(i)}) = \sum_{i=1}^{m} \Bigl[\log p(x^{(i)} \mid y^{(i)}) + \log p(y^{(i)})\Bigr]$$

Maximising with respect to each parameter separately gives **closed-form** solutions:

$$\boxed{\phi = \frac{1}{m}\sum_{i=1}^{m} \mathbf{1}\{y^{(i)} = 1\}} \quad \leftarrow \text{fraction of class-1 examples}$$

$$\boxed{\mu_k = \frac{\sum_{i=1}^{m} \mathbf{1}\{y^{(i)} = k\}\, x^{(i)}}{\sum_{i=1}^{m} \mathbf{1}\{y^{(i)} = k\}}} \quad \leftarrow \text{sample mean of class } k$$

$$\boxed{\Sigma = \frac{1}{m}\sum_{i=1}^{m} \bigl(x^{(i)} - \mu_{y^{(i)}}\bigr)\bigl(x^{(i)} - \mu_{y^{(i)}}\bigr)^T} \quad \leftarrow \text{pooled sample covariance}$$

**Derivation sketch for $\mu_1$:** the log-likelihood terms involving $\mu_1$ are:

$$\sum_{i:\,y^{(i)}=1} -\frac{1}{2}(x^{(i)}-\mu_1)^T\Sigma^{-1}(x^{(i)}-\mu_1)$$

Taking the gradient w.r.t. $\mu_1$ (using $\nabla_{\mu} (x-\mu)^T\Sigma^{-1}(x-\mu) = -2\Sigma^{-1}(x-\mu)$) and setting to zero gives $\mu_1 = $ sample mean of class-1 examples.

In [ ]:
class GDA:
    """Binary GDA with shared covariance (= LDA)."""

    def fit(self, X, y):
        m = len(y)
        # MLE for phi, mu_0, mu_1
        self.phi_  = np.mean(y == 1)
        self.mu_   = {k: X[y == k].mean(axis=0) for k in (0, 1)}
        # pooled covariance: subtract each point's class mean
        mu_yi = np.where(y[:, None] == 1, self.mu_[1], self.mu_[0])
        R = X - mu_yi
        self.Sigma_ = (R.T @ R) / m
        return self

    def _log_joint(self, X, k):
        prior = np.log(self.phi_) if k == 1 else np.log(1 - self.phi_)
        return multivariate_normal.logpdf(X, self.mu_[k], self.Sigma_) + prior

    def predict_proba(self, X):
        lp0 = self._log_joint(X, 0)
        lp1 = self._log_joint(X, 1)
        # p(y=1|x) via log-sum-exp
        return np.exp(lp1 - np.logaddexp(lp0, lp1))

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)


# ── generate synthetic data ─────────────────────────────────────────────────
rng = np.random.default_rng(0)
mu0_true    = np.array([-1., -1.])
mu1_true    = np.array([2., 1.])
Sigma_true  = np.array([[1.5, 0.5], [0.5, 1.0]])

m0, m1 = 100, 80
X0 = rng.multivariate_normal(mu0_true, Sigma_true, m0)
X1 = rng.multivariate_normal(mu1_true, Sigma_true, m1)
X_train = np.vstack([X0, X1])
y_train = np.hstack([np.zeros(m0), np.ones(m1)])

# ── fit and inspect estimates ────────────────────────────────────────────────
gda = GDA().fit(X_train, y_train)

print(f"phi       true={m1/(m0+m1):.3f}  est={gda.phi_:.3f}")
print(f"mu_0      true={mu0_true}  est={gda.mu_[0].round(3)}")
print(f"mu_1      true={mu1_true}   est={gda.mu_[1].round(3)}")
print(f"Sigma true:\n{Sigma_true}")
print(f"Sigma est:\n{gda.Sigma_.round(3)}")

## 5. The Decision Boundary

We predict class 1 when $p(y=1 \mid x) \geq 0.5$, i.e. when the log-odds are non-negative:

$$\log \frac{p(y=1\mid x)}{p(y=0\mid x)} = \log\frac{p(x\mid y=1)}{p(x\mid y=0)} + \log\frac{\phi}{1-\phi} \geq 0$$

Expanding the Gaussian log-likelihoods and cancelling $-\frac{1}{2}\log|\Sigma|$ terms (shared $\Sigma$):

$$= -\frac{1}{2}(x-\mu_1)^T\Sigma^{-1}(x-\mu_1) + \frac{1}{2}(x-\mu_0)^T\Sigma^{-1}(x-\mu_0) + \log\frac{\phi}{1-\phi}$$

Expanding and collecting terms **linear in $x$**:

$$= \underbrace{(\mu_1 - \mu_0)^T \Sigma^{-1}}_{\theta^T} x + \underbrace{\left(-\tfrac{1}{2}\mu_1^T\Sigma^{-1}\mu_1 + \tfrac{1}{2}\mu_0^T\Sigma^{-1}\mu_0 + \log\tfrac{\phi}{1-\phi}\right)}_{\theta_0} = \theta^T x + \theta_0$$

**This is linear in $x$** — GDA with shared $\Sigma$ always produces a linear (hyperplane) decision boundary.

**Example** — for $\mu_0=(-1,-1)$, $\mu_1=(2,1)$, $\Sigma = \begin{bmatrix}1.5&0.5\\0.5&1\end{bmatrix}$, $\phi=0.44$:

$$\theta = \Sigma^{-1}(\mu_1 - \mu_0) = \Sigma^{-1}\begin{bmatrix}3\\2\end{bmatrix}$$

The boundary is the set $\{x : \theta^T x + \theta_0 = 0\}$ — a line in $\mathbb{R}^2$.

In [ ]:
def decision_boundary_plot(model, X, y, ax, title):
    h = 0.04
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x1_min, x1_max, h),
                          np.arange(x2_min, x2_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]

    proba = model.predict_proba(grid)
    Z = (proba[:, 1] if proba.ndim == 2 else proba).reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=50, cmap="RdBu_r", alpha=0.25, vmin=0, vmax=1)
    ax.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=2)
    ax.scatter(X[y==0, 0], X[y==0, 1], c="steelblue", marker="o", s=25, alpha=0.7, label="Class 0")
    ax.scatter(X[y==1, 0], X[y==1, 1], c="tomato",    marker="s", s=25, alpha=0.7, label="Class 1")
    # plot class means
    ax.plot(*gda.mu_[0], "b*", ms=14)
    ax.plot(*gda.mu_[1], "r*", ms=14)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)


fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# our GDA
decision_boundary_plot(gda, X_train, y_train, axes[0],
    f"GDA (ours)  acc={np.mean(gda.predict(X_train)==y_train):.2f}")

# sklearn LDA as reference
lda_sk = LinearDiscriminantAnalysis().fit(X_train, y_train)
decision_boundary_plot(lda_sk, X_train, y_train, axes[1],
    f"sklearn LDA  acc={np.mean(lda_sk.predict(X_train)==y_train):.2f}")

plt.suptitle("GDA Decision Boundary", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# verify our theta matches the analytic formula
Sigma_inv = np.linalg.inv(gda.Sigma_)
theta_analytic = Sigma_inv @ (gda.mu_[1] - gda.mu_[0])
theta0_analytic = (-0.5 * gda.mu_[1] @ Sigma_inv @ gda.mu_[1]
                   + 0.5 * gda.mu_[0] @ Sigma_inv @ gda.mu_[0]
                   + np.log(gda.phi_ / (1 - gda.phi_)))
print(f"theta  = {theta_analytic.round(3)}")
print(f"theta0 = {theta0_analytic:.3f}")

## 6. GDA Implies Logistic Regression

The posterior $p(y=1 \mid x)$ from GDA is:

$$p(y=1 \mid x) = \frac{1}{1 + \exp(-(\theta^T x + \theta_0))}$$

with $\theta = \Sigma^{-1}(\mu_1 - \mu_0)$ — **exactly the logistic regression sigmoid**.

This means:

- **GDA $\Rightarrow$ logistic regression:** if $x \mid y$ is Gaussian (with shared $\Sigma$), the posterior is logistic
- **Logistic regression $\not\Rightarrow$ GDA:** logistic regression places no distributional assumption on $p(x \mid y)$

**Practical trade-off:**

| | GDA / LDA | Logistic Regression |
|---|---|---|
| Assumption | Gaussian class-conditionals | None on $p(x \mid y)$ |
| Estimation | Closed-form MLE | Iterative (gradient ascent) |
| Data efficiency | Better when Gaussian holds | More robust otherwise |
| Boundary | Always linear (LDA) | Linear |
| Prefer when | Continuous features, approx Gaussian | Binary / mixed / non-Gaussian features |

**Example** — when does logistic regression win? If class-conditional spreads differ along different axes (non-isotropic, non-equal), LDA's pooled-covariance assumption is violated and logistic regression is less biased.

In [ ]:
# GDA vs Logistic Regression on Gaussian and non-Gaussian data
rng2 = np.random.default_rng(7)

# non-Gaussian data: elongated blobs with very different orientations
Sigma_ng0 = np.array([[3., 0.], [0., 0.3]])
Sigma_ng1 = np.array([[0.3, 0.], [0., 3.]])
X_ng0 = rng2.multivariate_normal([-1, 0], Sigma_ng0, 150)
X_ng1 = rng2.multivariate_normal([1,  0], Sigma_ng1, 150)
X_ng  = np.vstack([X_ng0, X_ng1])
y_ng  = np.hstack([np.zeros(150), np.ones(150)])

gda_ng = GDA().fit(X_ng, y_ng)
lr_ng  = LogisticRegression(max_iter=1000).fit(X_ng, y_ng)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

decision_boundary_plot(gda_ng, X_ng, y_ng, axes[0],
    f"GDA — elongated classes\nacc={np.mean(gda_ng.predict(X_ng)==y_ng):.2f}")

# logistic boundary overlay
h = 0.04
x1r = np.arange(X_ng[:,0].min()-1, X_ng[:,0].max()+1, h)
x2r = np.arange(X_ng[:,1].min()-1, X_ng[:,1].max()+1, h)
xx, yy = np.meshgrid(x1r, x2r)
Z_lr = lr_ng.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:,1].reshape(xx.shape)

decision_boundary_plot(lr_ng, X_ng, y_ng, axes[1],
    f"Logistic Regression — same data\nacc={np.mean(lr_ng.predict(X_ng)==y_ng):.2f}")

plt.suptitle("GDA vs Logistic Regression", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Quadratic Discriminant Analysis (QDA)

Relax the shared-covariance assumption: let each class have its own $\Sigma_k$.

$$x \mid y=k \sim \mathcal{N}(\mu_k, \Sigma_k), \qquad k \in \{0,1\}$$

**MLE** for separate covariances (same $\hat\phi$, $\hat\mu_k$ as before):

$$\hat\Sigma_k = \frac{\sum_{i=1}^{m} \mathbf{1}\{y^{(i)}=k\}\,(x^{(i)}-\mu_k)(x^{(i)}-\mu_k)^T}{\sum_{i=1}^{m} \mathbf{1}\{y^{(i)}=k\}}$$

**Decision boundary** — the $|\Sigma_k|$ terms no longer cancel, giving a term **quadratic in $x$**:

$$\log\frac{p(y=1\mid x)}{p(y=0\mid x)} = -\frac{1}{2}(x-\mu_1)^T\Sigma_1^{-1}(x-\mu_1) + \frac{1}{2}(x-\mu_0)^T\Sigma_0^{-1}(x-\mu_0) - \frac{1}{2}\log\frac{|\Sigma_1|}{|\Sigma_0|} + \log\frac{\phi}{1-\phi}$$

**LDA vs QDA:**

| | LDA | QDA |
|---|---|---|
| Covariance | Shared $\Sigma$ | Separate $\Sigma_0, \Sigma_1$ |
| Boundary | Linear hyperplane | Quadratic surface |
| Extra parameters | $\frac{n(n+1)}{2}$ | $n(n+1)$ |
| Bias / Variance | Higher bias, lower variance | Lower bias, higher variance |
| Prefer when | Small dataset or similar class spreads | Large dataset and different class spreads |

In [ ]:
# LDA vs QDA when classes have different covariances
rng3 = np.random.default_rng(3)
Sigma0_q = np.array([[1., 0.], [0., 4.]])
Sigma1_q = np.array([[4., 0.], [0., 0.5]])
X_q0 = rng3.multivariate_normal([-1, 0], Sigma0_q, 200)
X_q1 = rng3.multivariate_normal([2,  0], Sigma1_q, 200)
X_q  = np.vstack([X_q0, X_q1])
y_q  = np.hstack([np.zeros(200), np.ones(200)])

lda_q = LinearDiscriminantAnalysis().fit(X_q, y_q)
qda_q = QuadraticDiscriminantAnalysis().fit(X_q, y_q)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, label in zip(axes,
        [lda_q, qda_q],
        ["LDA (shared $\\Sigma$)", "QDA (separate $\\Sigma_k$)"]):
    h = 0.05
    x1r = np.arange(X_q[:,0].min()-1, X_q[:,0].max()+1, h)
    x2r = np.arange(X_q[:,1].min()-1, X_q[:,1].max()+1, h)
    xx, yy = np.meshgrid(x1r, x2r)
    Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:,1].reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=50, cmap="RdBu_r", alpha=0.25, vmin=0, vmax=1)
    ax.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=2)
    ax.scatter(X_q[y_q==0,0], X_q[y_q==0,1], c="steelblue", s=15, alpha=0.5, label="Class 0")
    ax.scatter(X_q[y_q==1,0], X_q[y_q==1,1], c="tomato",    s=15, alpha=0.5, label="Class 1")
    acc = np.mean(model.predict(X_q) == y_q)
    ax.set_title(f"{label}  acc={acc:.2f}", fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle("LDA vs QDA — Different Class Covariances", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Practice Exercises

**Conceptual**

1. Show algebraically that when $\Sigma_0 = \Sigma_1 = \Sigma$, the decision boundary $p(y=1\mid x) = 0.5$ simplifies to $\theta^T x + \theta_0 = 0$ and derive $\theta = \Sigma^{-1}(\mu_1 - \mu_0)$.

2. The MLE estimate for $\Sigma$ uses $1/m$ rather than $1/(m-1)$. Is it biased? Does the bias matter asymptotically?

3. GDA requires $\Sigma \succ 0$ (positive definite). Why does this fail when $n > m$ and what can you do about it?

4. Suppose $x \mid y=k \sim \mathcal{N}(\mu_k, \Sigma)$ in $\mathbb{R}^n$. What is the dimension of the decision boundary hyperplane? Show that as $\phi \to 0$ the boundary shifts away from class-0.

**Numerical**

5. Implement QDA from scratch (separate $\Sigma_k$, `predict_proba` using log-odds) and verify the quadratic boundary appears on data with unequal covariances.

6. Compare GDA and logistic regression accuracy as a function of training set size (from 20 to 2000 samples). At what size does GDA's data-efficiency advantage peak?

7. Apply GDA to a real dataset (e.g. `sklearn.datasets.load_iris`, using only two classes). Visualize the class-conditional Gaussians and the decision boundary in PCA-projected 2D space.

**Reflection**

8. GDA, LDA, and logistic regression all produce linear decision boundaries (for LDA). What makes them fundamentally different? Under what conditions would you choose each?